### Get metadata records from iMOs catalogue

In [ ]:
import requests
import pandas as pd


In [ ]:
baseURL = "https://catalogue-imos.aodn.org.au/geonetwork/srv/eng/q?"
params = {
    "type": "dataset",
    "_content_type": "json",
    "fast": "index",
    "from": 1,
    "resultType": "details"
}

##https://catalogue-imos.aodn.org.au/geonetwork/srv/eng/q?title=NESP&type=dataset&_content_type=json&fast=index&from=1&resultType=details

In [ ]:
## make a query
response = requests.get(baseURL, params=params)
response.raise_for_status()

## parse the response
data = response.json()

## convert to pandas df
df = pd.DataFrame(data["metadata"])

## extract id and uuid from geonet:info dict
df["uuid"] = df["geonet:info"].apply(lambda x: x["uuid"])
df["id"] = df["geonet:info"].apply(lambda x: x["id"])


In [ ]:
df.head()

In [ ]:
data

In [6]:

## get the rest of the records itterating over the pages
## get the total numer of records
nRecords = int(data['summary']['@count'])


## itterate over the rest of the pages
page = 2
for i in range(1, int(nRecords/100) + 1):
    params["from"] = i*100 + 1
    response = requests.get(baseURL, params=params)
    response.raise_for_status()
    data = response.json()
    dfTemp = pd.DataFrame(data["metadata"])
    dfTemp["uuid"] = dfTemp["geonet:info"].apply(lambda x: x["uuid"])
    dfTemp["id"] = dfTemp["geonet:info"].apply(lambda x: x["id"])
    df = pd.concat([df, dfTemp], ignore_index=True)
    print(page)
    page += 1


2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17


In [7]:
df.columns

Index(['title', 'abstract', 'lineage', 'resourceConstraints', 'type',
       'legalConstraints', 'isHarvested', 'licenseLink', 'tempExtentBegin',
       'valid', 'tempExtentEnd', 'rating', 'source', 'category', 'status',
       'geoBox', 'recordOwner', 'defaultTitle', 'datasetLang', 'imageLink',
       'featureTypes', 'tempExtentPeriod', 'status_text', 'creationDate',
       'draft', 'groupOwner', '_locale', 'responsibleParty', 'displayOrder',
       'organisation', 'docLocale', 'popularity', 'useLimitation', 'keyword',
       'credit', 'publishedForGroup', 'otherConstr', 'image',
       'maintenanceAndUpdateFrequency_text', 'format', 'licenseName', 'root',
       'isTemplate', 'feedbackCount', 'owner', 'link', 'userinfo', 'topicCat',
       'jurisdictionLink', 'standardName', 'logo', 'parentId', 'keywordGroup',
       'geonet:info', 'identifier', 'parameter', 'classification_text',
       'platform', 'revisionDate', 'geoPolygon', 'boundingPolygon',
       'crsDetails', 'crs', 'uuid', 

In [8]:
## select columns
df = df[['id', 'uuid', 'category', 'type','title', 'abstract', 'keyword', 'geoPolygon', 'geoBox', 'tempExtentBegin', 'tempExtentEnd', 'creationDate']]


## Extract WKT geometries

Spatial extent information in the IMOS catalogue is spread across two fields — `geoPolygon` and `geoBox` — and neither is guaranteed to be present in every record. `geoPolygon` contains the actual polygon geometry as a WKT string or a list of WKT strings when multiple polygons are present. `geoBox` is a simpler bounding box encoded as a `minlon|minlat|maxlon|maxlat` string, used as a fallback when no polygon is available. Some records have both fields, some only one, and some have neither.

To produce a single, consistent `wkt` column we apply the following logic: if `geoPolygon` is present we use it (merging multiple polygons into a `MULTIPOLYGON`); otherwise we fall back to constructing a bounding-box polygon from `geoBox`; if neither field is available the record gets `None`.

In [9]:
## extract coordinates for geoBox
def geobox_to_wkt(geobox):
    if type(geobox) is str: 
        minlon, minlat, maxlon, maxlat = geobox.split("|")
        wkt = f"POLYGON(({minlon} {minlat}, {maxlon} {minlat}, {maxlon} {maxlat}, {minlon} {maxlat}, {minlon} {minlat}))"
        # print(wkt)
        return wkt
    if type(geobox) is list:
        temp = "|".join(geobox)
        if "POINT" in temp:
            # print("POINT")
            wkt = "MULTIPOINT("
            for i in range(len(geobox)):
                lon, lat = str(geobox[i]).split("|")
                wkt += f"({lon} {lat}),"
            wkt = wkt[:-1] + ")"
            return wkt
        else:
            # print ("POLYGON")
            wkt = "MULTIPOLYGON("
            for i in range(len(geobox)):
                print(geobox[i])
                minlon, minlat, maxlon, maxlat = str(geobox[i]).split("|")
                wkt += f"(({minlon} {minlat}, {maxlon} {minlat}, {maxlon} {maxlat}, {minlon} {maxlat}, {minlon} {minlat})),"
            wkt = wkt[:-1] + ")"    
            return wkt
    else:
        return None




In [10]:

df["wkt"] = None
for i in range(len(df)):
    print(i)
    if type(df['geoPolygon'][i]) is list: 
        wkt = "MULTIPOLYGON ("
        for j in range(len(df['geoPolygon'][i])):
            wktTemp = df['geoPolygon'][i][j].replace("MULTIPOLYGON (", "")
            wktTemp = wktTemp.replace("POLYGON", "")
            wktTemp = wktTemp.replace(")))", "))")
            wkt = wkt + wktTemp + ","
        wkt = wkt[:-1]
        df.loc[i, "wkt"] = wkt

    elif type(df['geoPolygon'][i]) is float:
        if type(df['geoBox'][i]) is float:
            df.loc[i, "wkt"] = None
        else:
            bbox = df['geoBox'][i]
            #if type(bbox) is list:
            #    kk = [bbox]
            wkt = geobox_to_wkt(bbox)
            df.loc[i, "wkt"] = wkt
    else:
        if df['geoPolygon'][i] == "MULTIPOLYGON EMPTY":
            df.loc[i, "wkt"] = None
        else:
            df.loc[i, "wkt"] = df['geoPolygon'][i]
        





0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
151.12|-34.16|151.25|-34.02
148.18|-42.70|148.30|-42.52
115.35|-32.08|115.45|-31.93
151.16|-34.19|151.27|-34.03
186
151.12|-34.16|151.25|-34.02
148.18|-42.70|148.30|-42.52
115.35|-32.08|115.45|-31.93
151.16|-34.19|151.27|-34.03
187
188
189
190
191
192
193
194
30.00|-62.00|180.00|25.00
-180.00|-62.00|-150.00|25.00
195
196
151.12|-34.16|151.25|-34.02
148.18|-42.70|1

In [11]:
df['keyword'].to_csv("keywords_20260507.csv")

In [12]:
## calculate the temporal extent in years decimal using temporalExtendBegin and temporalExtentEnd
df["temporalExtent"] = None
for i in range(len(df)):
    if df["tempExtentBegin"][i] is not None and df["tempExtentEnd"][i] is not None:
        tempExtentBegin = pd.to_datetime(df["tempExtentBegin"][i])
        tempExtentEnd = pd.to_datetime(df["tempExtentEnd"][i])
        tempExtent = tempExtentEnd - tempExtentBegin
        df.loc[i, "temporalExtent"] = tempExtent.days/365.25
    else:
        df.loc[i, "temporalExtent"] = None


In [13]:
## check for wrong temporal extents
df_wrongTextent = df.loc[df['temporalExtent'] <0]

## save to csv
if len(df_wrongTextent) > 0:
    df_wrongTextent.to_csv("IMOS_wrongTemporalExtent_20251106.csv")


In [14]:

## save to csv
df.to_csv("IMOS_datasets_20260507.csv", index=False)